# EOEPCA Operations Usage Notebook

The Operations Building Block is the observability stack for an EOEPCA deployment: Prometheus and Loki for metrics and logs, Grafana as the UI for both, and Alertmanager routing alerts to Keep for triage. Everything is wired together out of the box - dashboards, datasources and the alert pipeline all work with zero manual setup.

This notebook signs in to Grafana and Keep, queries live metrics and logs, checks the curated dashboards are loaded, and confirms an alert fired by Prometheus actually reaches Keep.

## Setup

In [ ]:
import base64
import json
import os
import shutil
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import requests
from IPython.display import Markdown, display

sys.path.append('../')
from modules.helpers import load_eoepca_state, oidc_session_login, test_cell, test_results

Load the `eoepca state` environment.

In [ ]:
load_eoepca_state()

In [ ]:
platform_domain = os.environ["INGRESS_HOST"]
http_scheme = os.environ["HTTP_SCHEME"]
operations_enable_iam = os.environ.get("OPERATIONS_ENABLE_IAM", "no") == "yes"

monitoring_url = f"{http_scheme}://monitoring.{platform_domain}"
alerting_url = f"{http_scheme}://alerting.{platform_domain}"

log_output_file = "operations_log.json"

print(f"Grafana: {monitoring_url}\nKeep: {alerting_url}\nIAM enabled: {operations_enable_iam}")

## Check the Building Block is up

Grafana and Keep each get their own ingress. A quick health check on both confirms the chart is deployed and reachable before doing anything else.

In [ ]:
# endpoints_healthy
grafana_response = requests.get(monitoring_url, timeout=10)
keep_response = requests.get(alerting_url, timeout=10)

print(f"Grafana: {grafana_response.status_code}")
print(f"Keep: {keep_response.status_code}")

assert grafana_response.status_code == 200
assert keep_response.status_code == 200

## Signing in

Grafana validates the OIDC login itself (`auth.generic_oauth`), and Keep sits behind its own `oauth2-proxy` sidecar Service which does the OIDC dance and forwards identity via `X-Forwarded-Email`/`X-Forwarded-Groups` headers. Both are app-level integrations rather than an ingress-layer plugin, so sign-in works identically under APISIX or NGINX.

`KEYCLOAK_TEST_ADMIN` is a member of the `monitoring-admin` and `alerting-admin` Keycloak groups created by the BB's IAM setup, so it comes back as a full admin on both UIs. If `OPERATIONS_ENABLE_IAM=no`, Grafana falls back to its chart-generated local admin account and Keep runs unauthenticated instead.

In [ ]:
# grafana_authentication
if operations_enable_iam:
    grafana_session = oidc_session_login(
        monitoring_url,
        "/login/generic_oauth",
        os.environ["KEYCLOAK_TEST_ADMIN"],
        os.environ["KEYCLOAK_TEST_PASSWORD"],
        session_cookie="grafana_session",
    )
else:
    if not shutil.which("kubectl"):
        raise RuntimeError("kubectl is required to fetch Grafana's local admin credentials when IAM is disabled")
    admin_user, admin_password = (
        base64.b64decode(subprocess.run(
            ["kubectl", "-n", "operations", "get", "secret", "kube-prometheus-stack-grafana", "-o", f"jsonpath={{.data.{key}}}"],
            capture_output=True, text=True, check=True,
        ).stdout).decode()
        for key in ("admin-user", "admin-password")
    )
    grafana_session = requests.Session()
    grafana_session.auth = (admin_user, admin_password)

grafana_user = grafana_session.get(f"{monitoring_url}/api/user", timeout=10).json()
print(f"Authenticated as {grafana_user['login']} (isGrafanaAdmin={grafana_user.get('isGrafanaAdmin')})")
assert grafana_user.get("isGrafanaAdmin") is True

In [ ]:
# keep_authentication
if operations_enable_iam:
    keep_session = oidc_session_login(
        alerting_url,
        "/",
        os.environ["KEYCLOAK_TEST_ADMIN"],
        os.environ["KEYCLOAK_TEST_PASSWORD"],
        session_cookie="keep",
    )
else:
    keep_session = requests.Session()
    keep_session.headers["x-api-key"] = "notebook-demo"  # NO_AUTH mode still needs *a* key header

alerts_response = keep_session.get(f"{alerting_url}/v2/alerts", timeout=10)
print(f"Keep alerts endpoint: {alerts_response.status_code}")
assert alerts_response.status_code == 200

## Exploring metrics and logs

Grafana ships with Prometheus and Loki pre-wired as datasources - no manual "add datasource" step needed.

In [ ]:
# grafana_datasources
datasources = grafana_session.get(f"{monitoring_url}/api/datasources", timeout=10).json()
by_type = {ds["type"]: ds for ds in datasources}
print("Datasources:", ", ".join(f"{ds['name']} ({ds['type']})" for ds in datasources))

assert "prometheus" in by_type
assert "loki" in by_type

for ds_type in ("prometheus", "loki"):
    health = grafana_session.get(f"{monitoring_url}/api/datasources/uid/{by_type[ds_type]['uid']}/health", timeout=10).json()
    print(f"{ds_type}: {health['status']} - {health['message']}")
    assert health["status"] == "OK"

prometheus_uid = by_type["prometheus"]["uid"]
loki_uid = by_type["loki"]["uid"]

Run the same PromQL query the "Kubernetes / Cluster View" dashboard uses for its headline panel, through Grafana's datasource proxy.

In [ ]:
# prometheus_query
cpu_expr = 'sum(rate(node_cpu_seconds_total{job="node-exporter", mode!="idle"}[5m])) / sum(rate(node_cpu_seconds_total{job="node-exporter"}[5m]))'
query = {
    "queries": [{"refId": "A", "datasource": {"uid": prometheus_uid}, "expr": cpu_expr, "instant": True}],
    "from": "now-5m",
    "to": "now",
}
response = grafana_session.post(f"{monitoring_url}/api/ds/query", json=query, timeout=10)
response.raise_for_status()
cpu_utilisation = response.json()["results"]["A"]["frames"][0]["data"]["values"][1][0]
print(f"Cluster CPU utilisation: {cpu_utilisation:.1%}")
assert 0 <= cpu_utilisation <= 1

Alloy tails every pod's stdout across the cluster and ships it to Loki, tagged with `namespace`/`pod`/`container` labels. Pull a few real lines from the Operations namespace's own logs - the same query you'd type into Grafana's Explore tab:

```logql
{namespace="operations"}
```

In [ ]:
# loki_query
query = {
    "queries": [{"refId": "A", "datasource": {"uid": loki_uid}, "expr": '{namespace="operations"}', "queryType": "range"}],
    "from": "now-15m",
    "to": "now",
}
response = grafana_session.post(f"{monitoring_url}/api/ds/query", json=query, timeout=10)
response.raise_for_status()
frame = response.json()["results"]["A"]["frames"][0]
fields = {f["name"]: i for i, f in enumerate(frame["schema"]["fields"])}
lines = frame["data"]["values"][fields["Line"]]
pods = frame["data"]["values"][fields["labels"]]

for pod, line in list(zip(pods, lines))[:3]:
    print(f"[{pod.get('pod', '?')}] {line[:120]}")

assert len(lines) > 0

## Dashboards

Five curated dashboards are shipped as labelled ConfigMaps and picked up automatically by Grafana's sidecar - covering cluster, node and pod resource usage, a Prometheus overview, and STAC API SLOs.

In [ ]:
# dashboards_loaded
dashboards = grafana_session.get(f"{monitoring_url}/api/search", params={"query": ""}, timeout=10).json()
by_title = {d["title"]: d for d in dashboards}
print("Dashboards:", ", ".join(sorted(by_title)))

expected = {
    "Kubernetes / Cluster View",
    "Kubernetes / Physical View",
    "Kubernetes / Workload View",
    "Prometheus / Overview",
}
assert expected.issubset(by_title)

cluster_view_url = f"{monitoring_url}{by_title['Kubernetes / Cluster View']['url']}"
display(Markdown(f"See it live, with real data: [{cluster_view_url}]({cluster_view_url})"))

## Alerting pipeline

`Watchdog` is a synthetic alert the baseline rules keep firing permanently on purpose - it's a canary, not an incident: if it ever *stops* firing, that itself means something upstream (Prometheus, Alertmanager, or the routing between them) is broken. Seeing it "firing" in Keep proves the whole chain works: Prometheus evaluates the rule, Alertmanager routes it to the relay, the relay injects the headers Keep's oauth2-proxy expects, and Keep ingests it.

In [ ]:
# watchdog_alert_in_keep
deadline = time.time() + 60
alerts = []
while time.time() < deadline:
    alerts = keep_session.get(f"{alerting_url}/v2/alerts", timeout=10).json()
    if any(a["name"] == "Watchdog" for a in alerts):
        break
    time.sleep(5)

watchdog = next((a for a in alerts if a["name"] == "Watchdog"), None)
if watchdog:
    print(f"{watchdog['description']} (status={watchdog['status']}, firing since {watchdog['firingStartTime']})")

incident_url = f"{alerting_url}/incidents"
display(Markdown(f"See it live: [{incident_url}]({incident_url})"))

assert watchdog is not None
assert watchdog["status"] == "firing" 

## Optional: Data Access request metrics

If the Data Access BB is deployed, its `stac-auth-proxy` labels its request metrics by STAC operation (`landing`, `search`, `list_collections`, ...) rather than just exposing generic CPU/memory - Prometheus is already scraping it via the same `ServiceMonitor` mechanism as every other BB. Skipped if Data Access isn't present.

In [ ]:
eoapi_domain = f"{http_scheme}://eoapi.{platform_domain}"

def stac_operation_counts():
    query = {"queries": [{
        "refId": "A",
        "datasource": {"uid": prometheus_uid},
        "expr": 'sum by (operation) (http_requests_total{job="eoapi-stac-auth-proxy"})',
        "instant": True,
    }], "from": "now-5m", "to": "now"}
    response = grafana_session.post(f"{monitoring_url}/api/ds/query", json=query, timeout=10)
    response.raise_for_status()
    frames = response.json()["results"]["A"]["frames"]
    return {f["schema"]["fields"][1]["labels"]["operation"]: f["data"]["values"][1][0] for f in frames}

try:
    requests.get(f"{eoapi_domain}/stac/", timeout=5).raise_for_status()
    data_access_present = True
except requests.RequestException:
    data_access_present = False

In [ ]:
# stac_proxy_metrics
if data_access_present:
    before = stac_operation_counts()
    for path in ("", "conformance", "collections", "search"):
        requests.get(f"{eoapi_domain}/stac/{path}", timeout=10)

    # Prometheus scrapes stac-auth-proxy every 30s - poll rather than read a stale scrape.
    deadline = time.time() + 45
    added = {}
    while time.time() < deadline and len(added) <= 1:
        after = stac_operation_counts()
        added = {op: after[op] - before.get(op, 0) for op in after if after[op] > before.get(op, 0)}
        if len(added) <= 1:
            time.sleep(5)

    plt.figure(figsize=(5, 3))
    plt.bar(added.keys(), added.values())
    plt.ylabel("requests (this run)")
    plt.title("eoAPI STAC requests by operation")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()

    assert len(added) > 1
else:
    print("Data Access (eoAPI) not detected on this deployment - skipping.")

## Results

In [ ]:
if test_results:
    for test, result in test_results.items():
        print(f"{test}: {result['status']} - {result['message']}")
    json.dump(test_results, open(log_output_file, "w"), indent=2)